In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/kzming2007/Catholic_ML_final_project/refs/heads/main/dataset/ur3_cobotops.csv")

## RandomForestRegressor

In [5]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 사용할 데이터 준비 (결측치 제거 및 X, y 분리)
# 원본 데이터프레임이 'df' 변수에 담겨있다고 가정합니다.
df_clean = df.dropna().copy()

input_cols = [
    'Current_J0', 'Current_J1', 'Current_J2', 'Current_J3', 'Current_J4', 'Current_J5',
    'Speed_J0', 'Speed_J1', 'Speed_J2', 'Speed_J3', 'Speed_J4', 'Speed_J5'
]
X = df_clean[input_cols]
y = df_clean['Tool_current']

# Train / Test 세트로 분할 (8:2 비율)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 랜덤 포레스트 회귀 모델 및 탐색할 하이퍼파라미터 정의
rf_model = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

# 5-Fold 교차 검증을 통한 최적의 하이퍼파라미터 탐색 (모든 CPU 코어 사용)
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',

    verbose=0
)

print("--- 모델 학습 및 파라미터 튜닝 진행 중... ---")
start_time = time.time()
grid_search.fit(X_train, y_train)
print(f"튜닝 완료! (소요 시간: {time.time() - start_time:.2f}초)\n")

# GridSearch 수행 결과 중 상위 10개 모델 순위 출력
results_df = pd.DataFrame(grid_search.cv_results_)
top10_results = results_df.sort_values(by='rank_test_score').head(10)

print("--- [GridSearch 결과 상위 10개 모델] ---")
for i, row in top10_results.iterrows():
    rank = row['rank_test_score']
    params = row['params']
    # neg_mean_squared_error이므로 양수로 변환
    mse = -row['mean_test_score']
    print(f"순위 {rank} | MSE: {mse:.6f} | 파라미터: {params}")

# 가장 성능이 좋은 모델로 Test 데이터 예측 및 최종 성능 평가
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse_val = mean_squared_error(y_test, y_pred)
rmse_val = np.sqrt(mse_val)

print("\n--- [최종 Test 데이터 평가 결과] ---")
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"R2 Score: {r2:.6f}")
print(f"MSE:      {mse_val:.6f}")
print(f"RMSE:     {rmse_val:.6f}")

--- 모델 학습 및 파라미터 튜닝 진행 중... ---


KeyboardInterrupt: 